# TextMamba3D — A100 Full Training Pipeline (v4.4)

**Target:** Single A100 40GB (Colab)
**Architecture:** v4.4 — Sequential Cross-Attention (TextBraTS MICCAI'25 inspired)
**Baseline:** v4.1 Mean Dice delta = -0.02% (best historical text guidance result)
**Goal:** Achieve POSITIVE text guidance delta via Q/KV direction reversal

v4.4 changes vs v4.1/v4.2:
- **SeqCA**: Text=Q, Image=KV (Step 1) -> Image=Q, Refined=KV (Step 2) — replaces PixelTextCrossAttention
- **Multi-scale fusion at stages 1,2,3** (preserved from v4.1 best result)
- **Zero-init Step 2 out_proj** for identity-preserving start
- **No new losses** — standard Dice+CE+Edge (isolate architecture effect)
- **No PWAM, no T2V, no Necessity, no EmbPerturb** — clean minimal change

**Key insight (TextBraTS):** Text should be Query ("where am I relevant?") not KV.
TextBraTS achieved +1.5% Dice on same BraTS2020 with SeqCA vs SwinUNETR baseline.

**VRAM estimate:** ~34-38 GB (similar to v4.1; SeqCA doubles attention maps but they're small)
**Checkpoint incompatibility:** V4.2 checkpoints NOT compatible — must train from scratch.

In [4]:
# Mount Google Drive (run in Colab web UI if using VS Code plugin)
from google.colab import drive
drive.mount('/content/drive')

# Install packages (cached on Drive)
!nvidia-smi
!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache mamba-ssm causal-conv1d transformers nibabel tensorboard pyyaml tqdm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sun Mar 15 05:30:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|          

In [ ]:
import os, zipfile, shutil

REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CODE_ZIP = os.path.join(DRIVE_BASE, 'TextMamba3D_code.zip')
DRIVE_CODE_DIR = os.path.join(DRIVE_BASE, 'TextMamba3D_code')

# 获取代码：VS Code 插件同步 → Drive zip → Drive 文件夹
tm_file = os.path.join(REPO_DIR, 'models/textmamba3d.py')
if os.path.exists(tm_file):
    print(f'Local code available at {REPO_DIR} (VS Code plugin)')
elif os.path.exists(DRIVE_CODE_ZIP):
    print(f'Extracting code from {DRIVE_CODE_ZIP}...')
    os.makedirs(REPO_DIR, exist_ok=True)
    with zipfile.ZipFile(DRIVE_CODE_ZIP, 'r') as zf:
        zf.extractall(REPO_DIR)
    print(f'Extracted to {REPO_DIR}')
elif os.path.exists(DRIVE_CODE_DIR):
    print(f'Copying code from {DRIVE_CODE_DIR}...')
    shutil.copytree(DRIVE_CODE_DIR, REPO_DIR)
    print(f'Copied to {REPO_DIR}')
else:
    raise FileNotFoundError(
        f'Code not found. Please either:\n'
        f'  1. Use VS Code Colab plugin to sync local project\n'
        f'  2. Upload TextMamba3D_code.zip to {DRIVE_BASE} on Google Drive'
    )

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Extract BraTS data from Drive
DATA_ZIP = os.path.join(DRIVE_BASE, "TextBraTS_data.zip")
DATA_DIR = os.path.join(REPO_DIR, "data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData")

if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    if os.path.exists(DATA_ZIP):
        print(f"Extracting {DATA_ZIP}...")
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(os.path.dirname(DATA_DIR))
        if os.path.exists(DATA_DIR):
            print(f"Data extracted. Cases: {len(os.listdir(DATA_DIR))}")
        else:
            print(f"ERROR: Expected path not found after extraction: {DATA_DIR}")
            print("Actual contents:", os.listdir(os.path.dirname(DATA_DIR)))
    else:
        print(f"ERROR: {DATA_ZIP} not found on Drive")
else:
    print(f"Data already exists. Cases: {len(os.listdir(DATA_DIR))}")

# Count samples
if os.path.exists(DATA_DIR):
    cases = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"Total BraTS cases: {len(cases)}")

## 3. V4.4 Code Patches

**Minimal changes** — only 2 files patched:
1. `models/fusion.py` — append SequentialCrossAttention + MultiScaleSeqCA
2. `models/textmamba3d.py` — swap import (2-line change)

In [ ]:
import pathlib

# [v4.4-1] Append Sequential Cross-Attention classes to models/fusion.py
fusion_path = pathlib.Path('models/fusion.py')
content = fusion_path.read_text(encoding='utf-8')

if 'SequentialCrossAttention' in content:
    print("[v4.4-1] SeqCA already exists in fusion.py, skipping")
else:
    NL = chr(10)
    seqca_code = NL.join([
        "",
        "",
        "# ---------------------------------------------------------------------------",
        "# Sequential Cross-Attention (TextBraTS-inspired, MICCAI 2025)",
        "# ---------------------------------------------------------------------------",
        "",
        "class SequentialCrossAttention(nn.Module):",
        "    \"\"\"TextBraTS-style Sequential Cross-Attention for text-guided segmentation.",
        "",
        "    Two-step cross-attention that reverses the Q/KV direction:",
        "      Step 1 (T2I): Text=Q, Image=KV -> refined features (text-length)",
        "      Step 2 (I2T): Image=Q, Refined=KV -> joint features (image-length)",
        "",
        "    This ensures text actively \"asks\" the image where its descriptions are",
        "    relevant, rather than the image passively querying text tokens.",
        "    \"\"\"",
        "",
        "    def __init__(self, feat_dim: int, text_dim: int, num_heads: int = 4):",
        "        super().__init__()",
        "        assert feat_dim % num_heads == 0, \\",
        "            f\"feat_dim ({feat_dim}) must be divisible by num_heads ({num_heads})\"",
        "        self.num_heads = num_heads",
        "        self.head_dim = feat_dim // num_heads",
        "        self.scale = self.head_dim ** -0.5",
        "",
        "        # Project text to image feature dimension",
        "        self.text_proj = nn.Sequential(",
        "            nn.Linear(text_dim, feat_dim),",
        "            nn.LayerNorm(feat_dim),",
        "        )",
        "",
        "        # Step 1: Text queries Image (T2I)",
        "        self.t2i_norm_q = nn.LayerNorm(feat_dim)",
        "        self.t2i_norm_kv = nn.LayerNorm(feat_dim)",
        "        self.t2i_q = nn.Linear(feat_dim, feat_dim)",
        "        self.t2i_k = nn.Linear(feat_dim, feat_dim)",
        "        self.t2i_v = nn.Linear(feat_dim, feat_dim)",
        "        self.t2i_out = nn.Sequential(",
        "            nn.Linear(feat_dim, feat_dim),",
        "            nn.LayerNorm(feat_dim),",
        "        )",
        "",
        "        # Step 2: Image queries Refined (I2T)",
        "        self.i2t_norm_q = nn.LayerNorm(feat_dim)",
        "        self.i2t_norm_kv = nn.LayerNorm(feat_dim)",
        "        self.i2t_q = nn.Linear(feat_dim, feat_dim)",
        "        self.i2t_k = nn.Linear(feat_dim, feat_dim)",
        "        self.i2t_v = nn.Linear(feat_dim, feat_dim)",
        "        self.i2t_out = nn.Linear(feat_dim, feat_dim)",
        "",
        "        # Zero-init Step 2 output for identity-preserving start",
        "        nn.init.zeros_(self.i2t_out.weight)",
        "        nn.init.zeros_(self.i2t_out.bias)",
        "",
        "    def _multi_head_attn(",
        "        self,",
        "        q: torch.Tensor,",
        "        k: torch.Tensor,",
        "        v: torch.Tensor,",
        "        mask: torch.Tensor | None = None,",
        "    ) -> torch.Tensor:",
        "        \"\"\"Multi-head attention computation.\"\"\"",
        "        B, Nq, D = q.shape",
        "        H, hd = self.num_heads, self.head_dim",
        "",
        "        q = q.reshape(B, Nq, H, hd).transpose(1, 2)",
        "        k = k.reshape(B, -1, H, hd).transpose(1, 2)",
        "        v = v.reshape(B, -1, H, hd).transpose(1, 2)",
        "",
        "        attn = (q @ k.transpose(-2, -1)) * self.scale",
        "",
        "        if mask is not None:",
        "            attn = attn.masked_fill(",
        "                mask.unsqueeze(1).unsqueeze(2) == 0,",
        "                float('-inf'),",
        "            )",
        "",
        "        attn = attn.softmax(dim=-1)",
        "        attn = torch.nan_to_num(attn)",
        "",
        "        out = (attn @ v).transpose(1, 2).reshape(B, Nq, D)",
        "        return out",
        "",
        "    def forward(",
        "        self,",
        "        x: torch.Tensor,",
        "        text_feat: torch.Tensor,",
        "        text_mask: torch.Tensor | None = None,",
        "    ) -> torch.Tensor:",
        "        \"\"\"Forward: x=[B,N,D] image, text_feat=[B,M,D_text] -> [B,N,D].\"\"\"",
        "        residual = x",
        "",
        "        # Project text to image feature space",
        "        text_proj = self.text_proj(text_feat)",
        "",
        "        # Step 1: Text=Q, Image=KV",
        "        q1 = self.t2i_q(self.t2i_norm_q(text_proj))",
        "        k1 = self.t2i_k(self.t2i_norm_kv(x))",
        "        v1 = self.t2i_v(self.t2i_norm_kv(x))",
        "        refined = self.t2i_out(self._multi_head_attn(q1, k1, v1))",
        "",
        "        # Step 2: Image=Q, Refined=KV",
        "        q2 = self.i2t_q(self.i2t_norm_q(x))",
        "        k2 = self.i2t_k(self.i2t_norm_kv(refined))",
        "        v2 = self.i2t_v(self.i2t_norm_kv(refined))",
        "        joint = self.i2t_out(self._multi_head_attn(q2, k2, v2, text_mask))",
        "",
        "        return residual + joint",
        "",
        "",
        "class MultiScaleSeqCA(nn.Module):",
        "    \"\"\"Apply Sequential Cross-Attention at multiple encoder scales.\"\"\"",
        "",
        "    def __init__(self, stage_dims: list[int], text_dim: int, num_heads: int = 4):",
        "        super().__init__()",
        "        self.attn_layers = nn.ModuleList([",
        "            SequentialCrossAttention(dim, text_dim, num_heads=num_heads)",
        "            for dim in stage_dims",
        "        ])",
        "",
        "    def forward(",
        "        self,",
        "        features: list[torch.Tensor],",
        "        text_feat: torch.Tensor,",
        "        text_mask: torch.Tensor | None = None,",
        "    ) -> list[torch.Tensor]:",
        "        return [",
        "            attn(feat, text_feat, text_mask)",
        "            for attn, feat in zip(self.attn_layers, features)",
        "        ]",
    ])
    fusion_path.write_text(content + NL + seqca_code + NL, encoding='utf-8')
    print("[v4.4-1] Appended SequentialCrossAttention + MultiScaleSeqCA to fusion.py")
    print(f"  fusion.py is now {len((content + seqca_code).splitlines())} lines")

In [ ]:
import os, pathlib, shutil

# [v4.4-2] Overwrite textmamba3d.py with V4.4 version (MultiScaleSeqCA)
# Self-contained: uses absolute path, does not depend on Cell 2's os.chdir
REPO_DIR = '/content/TextMamba3D'
tm_path = pathlib.Path(REPO_DIR) / 'models' / 'textmamba3d.py'

NL = chr(10)
tm_content = NL.join([
    "# models/textmamba3d.py",
    '"""Text-guided 3D medical image segmentation with Mamba architecture."""',
    "",
    "from typing import Optional",
    "",
    "import torch",
    "import torch.nn as nn",
    "",
    "from .decoder_3d import MambaDecoder3D",
    "from .encoder_3d import MambaEncoder3D",
    "from .fusion import MultiScaleSeqCA",
    "from .text_encoder import TextMambaEncoder",
    "",
    "",
    "class TextMamba3D(nn.Module):",
    '    """Text-guided 3D medical image segmentation model using Mamba architecture."""',
    "",
    "    def __init__(",
    "        self,",
    "        img_size: tuple[int, int, int] = (96, 96, 96),",
    "        in_channels: int = 4,",
    "        out_channels: int = 4,",
    "        embed_dim: int = 96,",
    "        depths: list[int] = [2, 2, 2, 2],",
    "        patch_size: tuple[int, int, int] = (4, 4, 4),",
    "        text_embed_dim: int = 256,",
    "        text_max_len: int = 256,",
    "        text_depth: int = 4,",
    "        d_state: int = 16,",
    "        dropout: float = 0.0,",
    "        use_pretrained_text: bool = True,",
    "        unfreeze_text_layers: int = 0,",
    "        use_checkpoint: bool = False,",
    "        text_model_path: str | None = None,",
    "        deep_supervision: bool = False,",
    "    ) -> None:",
    "        super().__init__()",
    "",
    "        self.text_embed_dim = text_embed_dim",
    "        self.text_max_len = text_max_len",
    "        bottleneck_dim = embed_dim * (2 ** (len(depths) - 1))",
    "",
    "        self.img_encoder = MambaEncoder3D(",
    "            img_size=img_size,",
    "            in_channels=in_channels,",
    "            embed_dim=embed_dim,",
    "            depths=depths,",
    "            patch_size=patch_size,",
    "            d_state=d_state,",
    "            dropout=dropout,",
    "            use_checkpoint=use_checkpoint,",
    "        )",
    "",
    "        self.text_encoder = TextMambaEncoder(",
    "            embed_dim=text_embed_dim,",
    "            max_len=text_max_len,",
    "            depth=text_depth,",
    "            d_state=d_state,",
    "            dropout=dropout,",
    "            use_pretrained=use_pretrained_text,",
    "            unfreeze_last_n=unfreeze_text_layers,",
    "            model_path=text_model_path,",
    "        )",
    "",
    "        # Multi-scale cross-attention: text guides skip connections at stages 1,2,3",
    "        # Stage 0 excluded (32K tokens too expensive for cross-attention)",
    "        stage_dims = [embed_dim * (2 ** i) for i in range(1, len(depths))]",
    "        self.multi_scale_attn = MultiScaleSeqCA(",
    "            stage_dims=stage_dims,     # [96, 192, 384] for embed_dim=48",
    "            text_dim=text_embed_dim,   # 256",
    "            num_heads=4,               # head_dim varies per stage",
    "        )",
    "",
    "        self.decoder = MambaDecoder3D(",
    "            img_size=img_size,",
    "            patch_size=patch_size,",
    "            out_channels=out_channels,",
    "            embed_dim=embed_dim,",
    "            depths=depths,",
    "            d_state=d_state,",
    "            dropout=dropout,",
    "            use_checkpoint=use_checkpoint,",
    "            deep_supervision=deep_supervision,",
    "        )",
    "",
    "        self.img_proj = nn.Sequential(",
    "            nn.Linear(bottleneck_dim, text_embed_dim),",
    "            nn.LayerNorm(text_embed_dim),",
    "        )",
    "",
    "    def forward(",
    "        self,",
    "        img: torch.Tensor,",
    "        text_ids: Optional[torch.Tensor] = None,",
    "        attention_mask: Optional[torch.Tensor] = None,",
    "        return_features: bool = False,",
    "        use_text: bool = True,",
    "    ) -> torch.Tensor | tuple[",
    "        torch.Tensor,",
    "        Optional[torch.Tensor],",
    "        Optional[torch.Tensor],",
    "        Optional[torch.Tensor],",
    "    ]:",
    '        """Forward pass for text-guided 3D segmentation."""',
    "        img_features = self.img_encoder(img)",
    "",
    "        has_text = use_text and text_ids is not None",
    "        if has_text:",
    "            text_features = self.text_encoder(text_ids, attention_mask)",
    "            # Multi-scale fusion: stages 1,2,3 get text cross-attention",
    "            # Stage 0 stays raw (32K tokens, too expensive for cross-attention)",
    "            fused = self.multi_scale_attn(",
    "                img_features[1:], text_features, attention_mask",
    "            )",
    "            decoder_features = [img_features[0]] + fused",
    "        else:",
    "            # Bypass fusion entirely for text-free path",
    "            decoder_features = img_features",
    "",
    "        seg_output = self.decoder(decoder_features)",
    "",
    "        if not return_features:",
    "            return seg_output",
    "",
    "        if has_text:",
    "            # Contrastive: project fused bottleneck for alignment",
    "            pixel_feat = decoder_features[-1]",
    "            img_global = self.img_proj(pixel_feat.mean(dim=1))",
    "            text_global = self.text_encoder.get_global_feature(text_features)",
    "            return seg_output, img_global, text_global, pixel_feat",
    "        else:",
    "            return seg_output, None, None, None",
    "",
    "    def forward_without_text(self, img: torch.Tensor) -> torch.Tensor:",
    '        """Convenience method for inference without text guidance."""',
    "        return self.forward(img, text_ids=None, use_text=False)",
    "",
])

tm_path.write_text(tm_content, encoding='utf-8')

# Clear __pycache__ to force Python to reload from .py
cache_dir = pathlib.Path(REPO_DIR) / 'models' / '__pycache__'
if cache_dir.exists():
    shutil.rmtree(cache_dir)
    print("[v4.4-2] Cleared models/__pycache__")

# Verify: read back and check critical lines
written = tm_path.read_text(encoding='utf-8')
assert 'MultiScaleSeqCA' in written, "FAIL: MultiScaleSeqCA not found in written file"
assert 'img_dim' not in written, "FAIL: old 'img_dim' kwarg still present"
assert 'MambaFusion' not in written, "FAIL: old MambaFusion reference still present"
assert 'self.multi_scale_attn' in written, "FAIL: self.multi_scale_attn not found"
print(f"[v4.4-2] Wrote {tm_path} ({len(written.splitlines())} lines)")
print(f"  Path: {tm_path.resolve()}")
print("  Verified: MultiScaleSeqCA, no img_dim, no MambaFusion, self.multi_scale_attn")
print("  OK — ready for smoke test")

In [ ]:
import pathlib

# [v4.4-3] Create configs/textbrats_v6.yaml
# Clean config: no PWAM/T2V/Necessity fields
# Phase 1: SeqCA only, contrastive_weight=0.0 to isolate architecture effect
NL = chr(10)
config_content = NL.join([
    "# textbrats_v6.yaml - V4.4 SeqCA (TextBraTS-inspired)",
    "# A100 40GB optimized | Phase 1: isolate SeqCA effect",
    "",
    "data:",
    "  data_dir: \"./data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData\"",
    "  dataset_type: \"textbrats\"",
    "  patch_size: [128, 128, 128]",
    "  batch_size: 4",
    "  num_workers: 4",
    "  train_ratio: 0.596",
    "  val_ratio: 0.149",
    "",
    "model:",
    "  img_size: [128, 128, 128]",
    "  in_channels: 4",
    "  out_channels: 4",
    "  embed_dim: 48",
    "  depths: [2, 2, 2, 2]",
    "  dropout: 0.1",
    "  text_embed_dim: 256",
    "  text_max_len: 256",
    "  use_pretrained_text: true",
    "  unfreeze_text_layers: 2",
    "  text_model_path: null",
    "",
    "loss:",
    "  dice_weight: 1.0",
    "  ce_weight: 1.0",
    "  edge_weight: 1.0",
    "  contrastive_weight: 0.0",
    "  temperature: 0.07",
    "  class_weights: [0.25, 3.0, 1.0, 4.0]",
    "",
    "augmentation:",
    "  use_elastic: true",
    "  use_modality_dropout: true",
    "",
    "training:",
    "  epochs: 200",
    "  lr: 0.0001",
    "  weight_decay: 0.01",
    "  warmup_epochs: 10",
    "  patience: 40",
    "  gradient_accumulation: 1",
    "  gradient_checkpointing: true",
    "  deep_supervision: true",
    "  ds_weights: [0.2, 0.1, 0.05]",
    "  use_amp: true",
    "  no_text_ratio: 0.15",
    "  gradient_clip_norm: 1.0",
    "",
    "eval:",
    "  metrics: [\"dice\", \"hd95\"]",
    "  sliding_window: true",
    "  sw_overlap: 0.5",
    "  sw_batch_size: 2",
    "",
    "experiment:",
    "  name: \"TextMamba3D_A100_v4.4_seqca\"",
    "  description: \"v4.4: SeqCA (Text=Q) at stages 1,2,3 — minimal change from v4.1\"",
])

pathlib.Path("configs/textbrats_v6.yaml").write_text(config_content + NL, encoding="utf-8")
print("[v4.4-3] Created configs/textbrats_v6.yaml")
print()
print("All V4.4 patches applied!")
print("Modified: fusion.py (appended SeqCA), textmamba3d.py (complete overwrite)")
print("Created: textbrats_v6.yaml")
print("Unchanged: train.py, losses/, decoder_3d.py, encoder_3d.py")

## 4. Smoke Test

Quick architecture check: 10 samples, 2 epochs.
Verifies SeqCA dimensions are correct and no OOM.

In [ ]:
# Smoke test: 10 samples, 2 epochs, vision-only (fastest check)
!python train.py --config configs/textbrats_v6.yaml --max-samples 10 --no-text-ratio 1.0 --grad-accum 1 --max-epochs 2

# Check VRAM
!nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits
print("(Peak VRAM after vision-only smoke test)")

In [ ]:
# Smoke test with text guidance: verify SeqCA fusion path works
!python train.py --config configs/textbrats_v6.yaml --max-samples 10 --no-text-ratio 0.0 --grad-accum 1 --max-epochs 2

!nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits
print("(Peak VRAM after text-guided smoke test -- should be slightly higher)")

## 5. Training

V4.4 uses the **original V4.1/V4.2 train.py** unchanged.
- Standard Dice + CE + Edge loss
- 15% no-text-ratio for robustness
- Warmup 10 epochs + cosine decay
- Early stopping patience 40

In [ ]:
import os, shutil, glob

DRIVE_CKPT = os.path.join(DRIVE_BASE, "checkpoints")
os.makedirs(DRIVE_CKPT, exist_ok=True)

def sync_checkpoints_to_drive():
    local_ckpt = os.path.join(REPO_DIR, "checkpoints")
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, "*.pth")):
        dst = os.path.join(DRIVE_CKPT, os.path.basename(f))
        shutil.copy2(f, dst)
    print(f"Synced checkpoints to {DRIVE_CKPT}")

# Clean old checkpoints (V4.4 incompatible with V4.2/V4.3)
for f in glob.glob(os.path.join(REPO_DIR, "checkpoints/*.pth")):
    os.remove(f)
for f in glob.glob(os.path.join(DRIVE_CKPT, '*.pth')):
    base = os.path.basename(f)
    if not base.startswith("best_v4."):
        os.remove(f)
print("Cleaned old checkpoints (V4.4 architecture change)")

In [ ]:
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

# V4.4 training: SeqCA fusion (Text=Q, Image=KV)
# Pipe output to log file to avoid Colab output truncation
!python -u train.py \
    --config configs/textbrats_v6.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 1 \
    2>&1 | tee training_v4.4.log | grep -E "^(Epoch [0-9]+:|  |Train|Best|Warning|Error|Traceback)"

# Sync and save
sync_checkpoints_to_drive()

best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.4.pth")
local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
if os.path.exists(local_best):
    shutil.copy2(local_best, best_ckpt)
    print(f"Best checkpoint saved: {best_ckpt}")

## 6. Full-Volume Evaluation (Sliding Window)

Compare with-text vs without-text on test set (94 cases).
Key metrics: per-region Dice (ET, WT, TC) and HD95.
**Success criterion:** with-text Dice > without-text Dice (positive delta).

In [ ]:
os.chdir(REPO_DIR)

ckpt = os.path.join(REPO_DIR, "checkpoints/best.pth")
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, "best_v4.4.pth")

if os.path.exists(ckpt):
    print("=" * 60)
    print("Evaluation: With Text (SeqCA fusion)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v6.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --use-text \
        --overlap 0.5

    print()

    print("=" * 60)
    print("Evaluation: Without Text (fusion bypassed)")
    print("=" * 60)
    !python evaluate_full.py \
        --config configs/textbrats_v6.yaml \
        --checkpoint "{ckpt}" \
        --split test \
        --no-text \
        --overlap 0.5

    print()
    print("=" * 60)
    print("Compare: with-text Dice - without-text Dice = text guidance delta")
    print("V4.1 baseline delta: -0.02%")
    print("V4.4 target: POSITIVE delta (SeqCA Text=Q should help)")
    print("=" * 60)
else:
    print(f"No checkpoint found at {ckpt}")
    print("Run training first (Cell 12)")

In [ ]:
import matplotlib.pyplot as plt

# Parse TensorBoard logs for training curves
try:
    from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
    ea = EventAccumulator('logs')
    ea.Reload()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: Training loss
    if 'Loss/train' in ea.Tags()['scalars']:
        steps = [s.step for s in ea.Scalars('Loss/train')]
        values = [s.value for s in ea.Scalars('Loss/train')]
        axes[0].plot(steps, values, label='Train Loss')
        axes[0].set_title('Training Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].legend()

    # Plot 2: Validation Dice (with vs without text)
    for tag, label, color in [
        ('Dice/val', 'With Text', 'blue'),
        ('Dice/val_no_text', 'Without Text', 'orange'),
    ]:
        if tag in ea.Tags()['scalars']:
            steps = [s.step for s in ea.Scalars(tag)]
            values = [s.value for s in ea.Scalars(tag)]
            axes[1].plot(steps, values, label=label, color=color)
    axes[1].set_title('Validation Dice (Text vs No-Text)')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    # Plot 3: Text guidance delta
    if 'Dice/val' in ea.Tags()['scalars'] and 'Dice/val_no_text' in ea.Tags()['scalars']:
        wt = {s.step: s.value for s in ea.Scalars('Dice/val')}
        nt = {s.step: s.value for s in ea.Scalars('Dice/val_no_text')}
        common = sorted(set(wt) & set(nt))
        deltas = [wt[e] - nt[e] for e in common]
        axes[2].plot(common, deltas, color='green')
        axes[2].axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Zero line')
        axes[2].set_title('Text Guidance Delta (with - without)')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Dice Delta')
        axes[2].legend()

    plt.tight_layout()
    plt.savefig('v4.4_training_curves.png', dpi=150)
    plt.show()
    print("Saved: v4.4_training_curves.png")

except Exception as e:
    print(f"Could not plot: {e}")
    print("TensorBoard logs may not exist yet. Run training first.")

## 7. Resume Training (After Colab Disconnect)

Run this cell to resume from the last checkpoint saved on Drive.

In [ ]:
import os, shutil
os.chdir(REPO_DIR)
os.environ["DRIVE_CKPT_DIR"] = DRIVE_CKPT

resume_ckpt = os.path.join(DRIVE_CKPT, "last.pth")
if os.path.exists(resume_ckpt):
    print(f"Resuming from {resume_ckpt}")
    !python train.py \
        --config configs/textbrats_v6.yaml \
        --resume "{resume_ckpt}" \
        --no-text-ratio 0.15 \
        --grad-accum 1

    sync_checkpoints_to_drive()

    best_ckpt = os.path.join(DRIVE_CKPT, "best_v4.4.pth")
    local_best = os.path.join(REPO_DIR, "checkpoints/best.pth")
    if os.path.exists(local_best):
        shutil.copy2(local_best, best_ckpt)
        print(f"Best checkpoint saved: {best_ckpt}")
else:
    print("No checkpoint to resume from.")
    print(f"Expected: {resume_ckpt}")
    print("Run training first (Cell 12)")